# Fire stream (NASA VIIRS): Person C
Turns the NASA FIRMS fire points into daily fire features per city (one row per city per day).
Only 2022 and 2023 are used. The 2024 files are kept sealed for the Week 14 extension and are never read here.
Every removal of data is also written down in `DECISIONS.md`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CITY_COUNTRY, CITY_COORDS, NASA_FILES, nasa_path

OUT_DIR = ROOT / "Dataset" / "Exported_Fire"
OUT_DIR.mkdir(parents=True, exist_ok=True)

START, END = pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31")   # same window as AQ and weather
RADII_KM = [50, 100, 150, 200, 300]
CONES = {"up30": (0, 30), "up45": (0, 45), "up60": (0, 60), "down45": (180, 45)}   # name: (offset from wind, half-angle)

audit = []   # one entry per filter step, copied into DECISIONS.md
def log_step(step, before, after, note=""):
    audit.append({"step": step, "rows_before": before, "rows_after": after,
                  "removed": before - after, "note": note})
    print(f"{step:45s} {before:>9,} -> {after:>9,}  (removed {before - after:,})")

**Setup.** Imports the shared settings from `src/config.py`. `NASA_FILES` lists only the 2022 and 2023 files, so 2024 cannot be loaded by mistake.
`CONES` holds the cone settings: three upwind half-angles (30, 45, 60 degrees) and one downwind placebo cone (wind direction + 180 degrees, 45 degrees wide).
`log_step` prints and stores the row count before and after every filter, for the bookkeeping file.

#### Step 1: Load and stack the 4 fire files

In [2]:
tables = []
for (country, year), name in NASA_FILES.items():
    # acq_time is text like "0833": read it as text, or the leading zero is lost
    df = pd.read_csv(ROOT / nasa_path(country, year), dtype={"acq_time": str})
    df["origin_country"] = country        # India file = IN, Pakistan file = PK
    df["source_file"] = name
    print(f"{name:26s} rows={len(df):8,}  first={df['acq_date'].min()}  last={df['acq_date'].max()}")
    tables.append(df)

fires_raw = pd.concat(tables, ignore_index=True)
print("\nTotal rows:", f"{len(fires_raw):,}")
print("Columns:", list(fires_raw.columns))
print("Duplicate points (same place, date and time):",
      fires_raw.duplicated(["latitude", "longitude", "acq_date", "acq_time"]).sum())
print("Satellites:", fires_raw["satellite"].value_counts().to_dict())
print("acq_time lengths:", fires_raw["acq_time"].str.len().value_counts().to_dict())

NASA - 2022 - India.csv    rows= 591,594  first=2022-01-01  last=2022-12-31


NASA - 2023 - India.csv    rows= 579,733  first=2023-01-01  last=2023-12-31
NASA - 2022.csv            rows=  60,973  first=2022-01-01  last=2022-12-31
NASA - 2023.csv            rows=  52,473  first=2023-01-01  last=2023-12-31



Total rows: 1,284,773
Columns: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type', 'origin_country', 'source_file']


Duplicate points (same place, date and time): 0
Satellites: {'N': 1284773}


acq_time lengths: {4: 1284773}


**Load and stack.** Reads the 4 files and adds `origin_country` from the file each point came from. That column is what later separates cross-border (India) fires from local (Pakistan) fires.
Checks: there are no duplicate points, and every point comes from one satellite (NOAA-20, code `N`), so no fire is counted twice by two satellites. Every `acq_time` kept its 4 digits.

#### Step 2: Keep only the study area

In [3]:
# The India file covers all of India (down to 8 N). Keep a box around the 11 cities
# with roughly 300 km to spare on every side, which is our largest radius.
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 26, 37, 68, 79
in_box = fires_raw["latitude"].between(LAT_MIN, LAT_MAX) & fires_raw["longitude"].between(LON_MIN, LON_MAX)
fires = fires_raw[in_box].copy()
log_step("Bounding box 26-37 N, 68-79 E", len(fires_raw), len(fires))
print(fires.groupby("origin_country").size())

Bounding box 26-37 N, 68-79 E                 1,284,773 ->   287,323  (removed 997,450)
origin_country
IN    185393
PK    101930
dtype: int64


**Study area.** Most of the India file is far from Punjab (southern and eastern India), so it is cut to a box around the 11 cities. The box is wide enough that every point within 300 km of a city is kept. The 300 km check comes in Step 6.

#### Step 3: Keep only vegetation fires (type 0)

In [4]:
print("Points by type before filtering:")
print(pd.crosstab(fires["type"], fires["origin_country"]))
before = len(fires)
fires = fires[fires["type"] == 0].copy()
log_step("Keep type 0 (vegetation fire)", before, len(fires), "drops type 2 static sources and type 3 offshore")

Points by type before filtering:
origin_country      IN     PK
type                         
0               182488  97480
2                 2891   4414
3                   14     36


Keep type 0 (vegetation fire)                   287,323 ->   279,968  (removed 7,355)


**Type filter.** NASA marks each point as 0 = vegetation fire, 2 = static land source (a hot spot that keeps returning to the same place: brick kilns, factories, flares), and 3 = offshore.
Our hypotheses are about crop burning, so only type 0 is kept.
Type 2 points burn at about the same rate every month, so they cannot explain day-to-day smog changes, and satellites only see the hottest kilns.
This removal is recorded in `DECISIONS.md`.

#### Step 4: Confidence filter (main version + two sensitivity versions)

In [5]:
month = pd.to_datetime(fires["acq_date"]).dt.month
print("Share of each confidence level by month (rows = month):")
print(pd.crosstab(month, fires["confidence"], normalize="index").round(2))
print("\nConfidence vs day/night:")
print(pd.crosstab(fires["confidence"], fires["daynight"]))

before = len(fires)
fires_nhl = fires.copy()                                  # sensitivity: everything, including low
fires = fires[fires["confidence"].isin(["n", "h"])].copy()  # MAIN version
fires_h = fires[fires["confidence"] == "h"].copy()        # sensitivity: high only
log_step("Keep confidence n and h (drop l)", before, len(fires), "main version")
print(f"Sensitivity versions kept: n+h+l = {len(fires_nhl):,} points, h only = {len(fires_h):,} points")

Share of each confidence level by month (rows = month):
confidence     h     l     n
acq_date                    
1           0.02  0.01  0.97
2           0.03  0.04  0.93
3           0.03  0.09  0.88
4           0.05  0.27  0.68
5           0.03  0.36  0.60
6           0.02  0.19  0.79
7           0.01  0.08  0.91
8           0.01  0.06  0.94
9           0.01  0.11  0.88
10          0.02  0.08  0.90
11          0.03  0.02  0.94
12          0.01  0.01  0.98

Confidence vs day/night:
daynight         D      N
confidence               
h             8459     75
l            41172      0
n           203671  26591


Keep confidence n and h (drop l)                279,968 ->   238,796  (removed 41,172)
Sensitivity versions kept: n+h+l = 279,968 points, h only = 8,534 points


**Confidence filter.** VIIRS gives every point a confidence code:
- `l` (low): daytime only, a weak heat signal, often sun glint or hot bare ground.
- `n` (nominal): a clear heat signal, the normal "real fire" case.
- `h` (high): the pixel is so hot the sensor saturates, which means a very intense fire.

The main version keeps `n` and `h`. Low-confidence points are common in April and May, when fields are hot and bare, and rare in October and November.
Two versions are kept for sensitivity checks: `n+h+l` and `h` only.

#### Step 5: Fix time (UTC to Pakistan time)

In [6]:
def to_local_date(df):
    # join date and 4-digit time into one UTC datetime, add 5 hours, THEN take the date
    utc = pd.to_datetime(df["acq_date"] + " " + df["acq_time"].str.zfill(4), format="%Y-%m-%d %H%M")
    df["time_utc"] = utc
    df["time_local"] = utc + pd.Timedelta(hours=5)
    df["date"] = df["time_local"].dt.normalize()
    return df

for name in ["fires", "fires_nhl", "fires_h"]:
    df = to_local_date(globals()[name])
    inside = df["date"].between(START, END)
    globals()[name] = df[inside].copy()
    if name == "fires":
        log_step("Drop points that fall on 2024-01-01 local time", len(df), int(inside.sum()),
                 "late 31 Dec 2023 UTC detections")

shifted = (fires["date"] != pd.to_datetime(fires["acq_date"])).sum()
print("Points whose local date differs from the UTC date:", f"{shifted:,}")
print("Local date range:", fires["date"].min().date(), "to", fires["date"].max().date())

Drop points that fall on 2024-01-01 local time   238,796 ->   238,756  (removed 40)


Points whose local date differs from the UTC date: 26,626
Local date range: 2022-01-01 to 2023-12-31


**Time.** NASA times are in UTC. The 5 hours are added to the full date and time *before* the date is taken. Shifting only the date column would put evening detections on the wrong day. This matches the AQ and weather notebooks.
A few detections on the evening of 31 Dec 2023 (UTC) fall on 1 Jan 2024 local time. They are outside our window and are dropped.

#### Step 6: 0/1 dummy columns for day/night and confidence

In [7]:
for df in (fires, fires_nhl, fires_h):
    df["is_night"] = (df["daynight"] == "N").astype(int)    # reference: day
    df["conf_high"] = (df["confidence"] == "h").astype(int)  # reference: nominal (or low)
    df["conf_low"] = (df["confidence"] == "l").astype(int)
print(fires[["daynight", "is_night", "confidence", "conf_high"]].drop_duplicates().sort_values(["is_night", "conf_high"]))

       daynight  is_night confidence  conf_high
38            D         0          n          0
126           D         0          h          1
387           N         1          n          0
265626        N         1          h          1


**Dummy encoding.** `daynight` and `confidence` are text categories. They become 0/1 columns, leaving out one reference level each: day for `daynight`, nominal for confidence. These are point-level columns. The daily tables use counts and FRP instead.

#### Step 7: Distance and bearing from every city to every fire

In [8]:
EARTH_R_KM = 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works on whole numpy arrays at once."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_R_KM * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    """Compass bearing FROM point 1 TO point 2: 0 = north, 90 = east."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    x = np.sin(lon2 - lon1) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lon2 - lon1)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

# toy checks: a point due north has bearing 0, due east about 90
assert abs(bearing_deg(31, 74, 32, 74) - 0) < 1e-6
assert abs(bearing_deg(31, 74, 31, 75) - 90) < 1
assert abs(haversine_km(31, 74, 32, 74) - 111.2) < 0.5

def city_fire_pairs(df, max_km=max(RADII_KM)):
    """One row per (city, fire) pair within max_km. Loops over the 11 cities, not over fires."""
    out = []
    lat, lon = df["latitude"].to_numpy(), df["longitude"].to_numpy()
    for city, (clat, clon) in CITY_COORDS.items():
        dist = haversine_km(clat, clon, lat, lon)
        near = dist <= max_km
        part = df.loc[near, ["date", "origin_country", "frp", "confidence"]].copy()
        part.insert(0, "city", city)
        part["dist_km"] = dist[near]
        part["bearing_deg"] = bearing_deg(clat, clon, lat[near], lon[near])
        out.append(part)
    return pd.concat(out, ignore_index=True)

pairs_nhl = city_fire_pairs(fires_nhl)      # all confidence levels; the other versions are filters of this
pairs = pairs_nhl[pairs_nhl["confidence"].isin(["n", "h"])].copy()
pairs_h = pairs_nhl[pairs_nhl["confidence"] == "h"].copy()
print("City-fire pairs within 300 km (main version):", f"{len(pairs):,}")

print("\nFires within 150 km and 300 km of each city, by origin (main version):")
check = pd.concat({r: pairs[pairs["dist_km"] <= r].groupby(["city", "origin_country"]).size().unstack(fill_value=0)
                   for r in (150, 300)}, axis=1).reindex(list(CITY_COORDS)).fillna(0).astype(int)
print(check)

City-fire pairs within 300 km (main version): 1,422,843

Fires within 150 km and 300 km of each city, by origin (main version):


                   150            300       
origin_country      IN     PK      IN     PK
city                                        
Lahore           61873  28523  125479  52819
Faisalabad        8108  39277   97859  65337
Multan               0  18414   16007  55248
Rawalpindi        1882  10884   17394  46112
Sialkot          24421  20061  105779  49145
Rahim Yar Khan      79   8425    3232  22740
Islamabad         1779  10338   14967  45092
Amritsar         66170  20462  129160  48181
Ferozepur        87790  19343  131143  49195
Ludhiana        105121    786  142361  31930
Jalandhar        91341   5683  137942  35721


**Distance and bearing.** For each city, the Haversine formula gives the distance to every fire, and `atan2` gives the compass bearing from the city to the fire. Both are computed for all fires at once (vectorised), one city at a time. Everything within 300 km is kept, so every smaller radius is just a filter on this table.
One fire can belong to several cities, which is correct: its smoke can reach any of them.

The table shows that India-origin fires within 150 km exist mainly for Lahore, Sialkot and Faisalabad. Multan and Rahim Yar Khan have essentially none, and Islamabad and Rawalpindi very few. That is why the 300 km radius is also kept.

#### Step 8: The cone function, tested on toy cases

In [9]:
def angle_diff(a, b):
    """Smallest difference between two compass angles, 0 to 180. 350 and 10 are 20 apart."""
    return np.abs((np.asarray(a) - np.asarray(b) + 180) % 360 - 180)

def cone_mask(bearing, wind_from, half_angle, offset=0):
    """True if a fire lies inside the cone.
    wind_from: Open-Meteo wind direction = where the wind comes FROM, so upwind fires lie at that bearing.
    offset=0 gives the upwind cone, offset=180 the downwind (placebo) cone."""
    return angle_diff(bearing, (np.asarray(wind_from) + offset) % 360) <= half_angle

# toy checks
assert angle_diff(350, 10) == 20 and angle_diff(10, 350) == 20 and angle_diff(0, 180) == 180
assert cone_mask(0, 0, 45)              # wind from north -> fire to the north is upwind
assert not cone_mask(180, 0, 45)        # wind from north -> fire to the south is not
assert cone_mask(180, 0, 45, offset=180)  # ...but it is in the downwind cone
assert cone_mask(350, 10, 30) and not cone_mask(350, 10, 15)   # wrap-around across 0/360
assert cone_mask(44, 0, 45) and not cone_mask(46, 0, 45)
print("All cone toy checks passed")

All cone toy checks passed


**Cone function.** A fire is upwind when its bearing from the city is within the half-angle of the direction the wind comes from. Open-Meteo reports where the wind comes *from*.
The difference between two angles is taken around the circle, so 350 and 10 degrees count as 20 apart, not 340.
The checks above confirm both rules, and the downwind version (direction + 180 degrees).

#### Step 9: Daily radius features on the full city-day grid

In [10]:
CITIES = list(CITY_COORDS)
DATES = pd.date_range(START, END, freq="D")
GRID = pd.MultiIndex.from_product([CITIES, DATES], names=["city", "date"])
print("Grid rows (11 cities x days):", len(GRID))

def daily_counts(p, cone_name):
    """Fire count and FRP sum per city-day, for every radius and origin (IN, PK, ALL). Long format."""
    rows = []
    for r in RADII_KM:
        sub = p[p["dist_km"] <= r]
        for origin in ["IN", "PK", "ALL"]:
            s = sub if origin == "ALL" else sub[sub["origin_country"] == origin]
            g = s.groupby(["city", "date"]).agg(fire_count=("frp", "size"), frp_sum_mw=("frp", "sum"))
            g = g.reindex(GRID, fill_value=0).reset_index()      # no fires that day -> 0, never missing
            g["radius_km"], g["origin"], g["cone"] = r, origin, cone_name
            rows.append(g)
    return pd.concat(rows, ignore_index=True)

long_radius = daily_counts(pairs, "none")
long_radius["fire_count"] = long_radius["fire_count"].astype(int)
print("Long radius table:", long_radius.shape)
print("Missing values:", long_radius.isna().sum().sum())

Grid rows (11 cities x days): 8030


Long radius table: (120450, 7)
Missing values: 0


**Radius features.** For every city, day, radius (50 to 300 km) and origin (India, Pakistan, both), this counts the fires and adds up their fire radiative power (FRP, in megawatts, a measure of fire intensity).
The table starts from a complete grid of 11 cities x 730 days = 8,030 city-days. A day without fires gets 0, never a missing value.

#### Step 10: Cone features (needs Person B's daily wind table)

In [11]:
# Person B's daily wind table. Expected columns: city, date, wind_dir_from_deg, stagnant (0/1)
WIND_FILE = ROOT / "Dataset" / "Exported_Weather" / "weather_daily_wind.csv"

def cone_counts(p, wind):
    """Long-format counts for every cone. On stagnant days the cone keeps every fire (= radius count)."""
    p = p.merge(wind[["city", "date", "wind_dir_from_deg", "stagnant"]], on=["city", "date"], how="left")
    out = []
    for name, (offset, half) in CONES.items():
        keep = cone_mask(p["bearing_deg"], p["wind_dir_from_deg"], half, offset) | (p["stagnant"] == 1)
        out.append(daily_counts(p[keep], name))
    return pd.concat(out, ignore_index=True)

if WIND_FILE.exists():
    wind = pd.read_csv(WIND_FILE, parse_dates=["date"])
    assert {"city", "date", "wind_dir_from_deg", "stagnant"} <= set(wind.columns), "wind file is missing columns"
    long_cone = cone_counts(pairs, wind)
    stagnant = wind.set_index(["city", "date"])["stagnant"].reindex(GRID)
    print("Cone table:", long_cone.shape)
else:
    long_cone, stagnant = None, None
    print(f"SKIPPED: {WIND_FILE.relative_to(ROOT)} not found yet (Person B). "
          "Rerun from this cell once it exists; radius features are complete without it.")

SKIPPED: Dataset/Exported_Weather/weather_daily_wind.csv not found yet (Person B). Rerun from this cell once it exists; radius features are complete without it.


**Cone features.** This uses the daily wind direction from Person B's weather table: the vector mean of the day's hourly winds, which is where the wind came *from*. It counts the fires inside each cone (30, 45 and 60 degrees upwind, plus the 45 degree downwind placebo).
On **stagnant days**, when there is almost no net wind, the daily direction is meaningless, so the cone keeps every fire in the radius and the `stagnant` flag marks those days. Leaving those days empty would drop the worst smog days later.
Until Person B's file exists, this cell prints a message and skips. Rerun the notebook once the file is in place.

#### Step 11: Long and wide daily tables

In [12]:
fire_long = pd.concat([long_radius] + ([long_cone] if long_cone is not None else []), ignore_index=True)
fire_long.insert(1, "country", fire_long["city"].map(CITY_COUNTRY))
fire_long = fire_long[["city", "country", "date", "cone", "radius_km", "origin", "fire_count", "frp_sum_mw"]]

def wide_from(long, cone, radius, prefix):
    """One column per origin for count and FRP, e.g. fire_cnt_r150_in."""
    s = long[(long["cone"] == cone) & (long["radius_km"] == radius)]
    w = s.pivot_table(index=["city", "date"], columns="origin", values=["fire_count", "frp_sum_mw"])
    w.columns = [f"{'fire_cnt' if v == 'fire_count' else 'fire_frp'}_{prefix}_{o.lower()}" for v, o in w.columns]
    return w.reindex(GRID)

# default setting: 150 km, no cone + 45 degree upwind cone + downwind placebo
parts = [wide_from(fire_long, "none", 150, "r150")]
if long_cone is not None:
    parts += [wide_from(fire_long, "up45", 150, "up45_r150"), wide_from(fire_long, "down45", 150, "down45_r150")]
# confidence sensitivity versions, 150 km, no cone
parts += [wide_from(daily_counts(pairs_h[pairs_h["dist_km"] <= 150], "none"), "none", 150, "r150_conf_h"),
          wide_from(daily_counts(pairs_nhl[pairs_nhl["dist_km"] <= 150], "none"), "none", 150, "r150_conf_nhl")]
fire_wide = pd.concat(parts, axis=1)
if stagnant is not None:
    fire_wide["stagnant"] = stagnant
fire_wide = fire_wide.reset_index()
fire_wide.insert(1, "country", fire_wide["city"].map(CITY_COUNTRY))
cnt_cols = [c for c in fire_wide.columns if c.startswith("fire_cnt")]
fire_wide[cnt_cols] = fire_wide[cnt_cols].astype(int)

print("Long table:", fire_long.shape, "  Wide table:", fire_wide.shape)
print(fire_wide.columns.tolist())

Long table: (120450, 8)   Wide table: (8030, 21)
['city', 'country', 'date', 'fire_cnt_r150_all', 'fire_cnt_r150_in', 'fire_cnt_r150_pk', 'fire_frp_r150_all', 'fire_frp_r150_in', 'fire_frp_r150_pk', 'fire_cnt_r150_conf_h_all', 'fire_cnt_r150_conf_h_in', 'fire_cnt_r150_conf_h_pk', 'fire_frp_r150_conf_h_all', 'fire_frp_r150_conf_h_in', 'fire_frp_r150_conf_h_pk', 'fire_cnt_r150_conf_nhl_all', 'fire_cnt_r150_conf_nhl_in', 'fire_cnt_r150_conf_nhl_pk', 'fire_frp_r150_conf_nhl_all', 'fire_frp_r150_conf_nhl_in', 'fire_frp_r150_conf_nhl_pk']


**Two daily tables.**
- `fire_long` holds every setting (cone x radius x origin). It is used for the cone and radius sensitivity analysis.
- `fire_wide` is one row per city per day with the default setting (150 km), ready to merge with AQ and weather on `city` + `date`. It also has the two confidence sensitivity versions at 150 km.

Lags (1 to 3 days) are not added here. They are added after the merge, grouped by city.

#### Step 12: Summary statistics for the default fire features

In [13]:
key_cols = ["fire_cnt_r150_in", "fire_cnt_r150_pk", "fire_cnt_r150_all",
            "fire_frp_r150_in", "fire_frp_r150_pk", "fire_frp_r150_all"]
summary = pd.DataFrame({
    "mean": fire_wide[key_cols].mean(),
    "median": fire_wide[key_cols].median(),
    "sd": fire_wide[key_cols].std(),
    "iqr": fire_wide[key_cols].quantile(0.75) - fire_wide[key_cols].quantile(0.25),
    "skewness": fire_wide[key_cols].skew(),
    "share_zero_days": (fire_wide[key_cols] == 0).mean(),
}).round(2)
print("All 11 cities, 2022-01-01 to 2023-12-31, 150 km, no cone:")
print(summary)

octnov = fire_wide[fire_wide["date"].dt.month.isin([10, 11])]
print("\nSame, Oct-Nov only (rice-burning season):")
print(octnov[key_cols].agg(["mean", "median", "std"]).T.round(2))

All 11 cities, 2022-01-01 to 2023-12-31, 150 km, no cone:
                     mean  median       sd     iqr  skewness  share_zero_days
fire_cnt_r150_in    55.86    0.00   224.39   10.00      6.43             0.52
fire_cnt_r150_pk    22.69    5.00    47.25   22.00      4.23             0.24
fire_cnt_r150_all   78.55   10.00   244.39   36.00      5.71             0.16
fire_frp_r150_in   376.43    0.00  1585.76   45.35      6.55             0.52
fire_frp_r150_pk   120.96   18.12   283.92  106.67      5.09             0.24
fire_frp_r150_all  497.40   43.04  1703.09  182.07      5.93             0.16

Same, Oct-Nov only (rice-burning season):
                      mean  median      std
fire_cnt_r150_in    218.45   17.00   456.52
fire_cnt_r150_pk     49.47   17.00    68.51
fire_cnt_r150_all   267.93   67.00   475.20
fire_frp_r150_in   1454.34   81.82  3186.81
fire_frp_r150_pk    248.84   73.56   370.20
fire_frp_r150_all  1703.18  351.82  3282.29


**Summary statistics.** Fire counts are zero on most city-days, and a few days have thousands of fires, so the distributions are extremely right-skewed.
The **median is 0** for most columns, so it says "a typical day has no fires nearby". The **mean** is pulled up by the burning-season peaks.
Neither measure describes these variables well on its own, which is why the share of zero days is reported too and why a log transform (log1p) is needed before modeling.

#### Step 13: Outlier audit (z-score rule and IQR rule)

In [14]:
def outlier_audit(s):
    z = (s - s.mean()) / s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    z_flag = z.abs() > 3
    iqr_flag = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    return z_flag, iqr_flag, {
        "z_outliers": int(z_flag.sum()), "iqr_outliers": int(iqr_flag.sum()),
        "upper_iqr_fence": q3 + 1.5 * iqr,
        "sd_all": s.std(), "sd_without_z_outliers": s[~z_flag].std(),
    }

rows = {}
for col in ["fire_cnt_r150_all", "fire_frp_r150_all", "fire_cnt_r150_in"]:
    for scale in ["raw", "log1p"]:
        s = fire_wide[col] if scale == "raw" else np.log1p(fire_wide[col])
        z_flag, iqr_flag, stats = outlier_audit(s)
        rows[(col, scale)] = stats
        if scale == "raw":       # keep flags, never delete: big fire days are real events
            fire_wide[f"out_z_{col}"] = z_flag.astype(int)
            fire_wide[f"out_iqr_{col}"] = iqr_flag.astype(int)
print(pd.DataFrame(rows).T.round(2))

                         z_outliers  iqr_outliers  upper_iqr_fence   sd_all  \
fire_cnt_r150_all raw         210.0        1083.0            92.00   244.39   
                  log1p         0.0          31.0             7.51     1.85   
fire_frp_r150_all raw         205.0        1121.0           460.04  1703.09   
                  log1p         0.0           0.0            10.44     2.42   
fire_cnt_r150_in  raw         193.0        1291.0            25.00   224.39   
                  log1p        85.0         337.0             5.99     1.90   

                         sd_without_z_outliers  
fire_cnt_r150_all raw                   106.54  
                  log1p                   1.85  
fire_frp_r150_all raw                   697.18  
                  log1p                   2.42  
fire_cnt_r150_in  raw                    88.66  
                  log1p                   1.81  


**Outlier audit.** Both rules are applied to the raw values and to log1p values:
- **z-score rule:** |z| > 3.
- **IQR rule:** outside Q1 - 1.5 x IQR or Q3 + 1.5 x IQR.

What it shows:
- **IQR rule on raw counts.** Most days are 0, so the IQR is tiny and the upper fence is very low. Almost every real burning day gets flagged. The rule is weak for zero-heavy data.
- **z-score rule on raw counts.** The huge peak days inflate the SD, compare `sd_all` with `sd_without_z_outliers`. A large SD hides moderately high days from the z rule. This is the *masking* effect.
- **On the log scale** both rules flag far fewer days.

Big fire days are the real events we study, so nothing is deleted. Flag columns (`out_z_...`, `out_iqr_...`) are added instead.

#### Step 14: Save and final checks

In [15]:
points_out = fires_nhl[["date", "time_utc", "time_local", "latitude", "longitude", "origin_country", "frp",
                        "confidence", "conf_high", "conf_low", "daynight", "is_night", "bright_ti4",
                        "bright_ti5", "scan", "track", "source_file"]]
points_out.to_csv(OUT_DIR / "fire_points_clean.csv", index=False)
fire_long.to_csv(OUT_DIR / "fire_city_day_long.csv", index=False)
fire_wide.to_csv(OUT_DIR / "fire_city_day_wide.csv", index=False)
pd.DataFrame(audit).to_csv(OUT_DIR / "fire_filter_audit.csv", index=False)

print("Saved to", OUT_DIR.relative_to(ROOT))
print("Cities:", fire_wide["city"].nunique(), "| rows:", len(fire_wide),
      "| duplicate city+date:", fire_wide.duplicated(["city", "date"]).sum(),
      "| missing in fire columns:", fire_wide[[c for c in fire_wide if c.startswith("fire_")]].isna().sum().sum())
print("Date range:", fire_wide["date"].min().date(), "to", fire_wide["date"].max().date())
on = fire_wide[fire_wide["date"].dt.month.isin([10, 11])]
print("\nShare of Oct-Nov days with at least one fire within 150 km (any origin):")
print((on.groupby("city")["fire_cnt_r150_all"].apply(lambda s: (s > 0).mean())).round(2).reindex(CITIES))
print("\nFilter audit:")
print(pd.DataFrame(audit).to_string(index=False))

Saved to Dataset/Exported_Fire
Cities: 11 | rows: 8030 | duplicate city+date: 0 | missing in fire columns: 0
Date range: 2022-01-01 to 2023-12-31

Share of Oct-Nov days with at least one fire within 150 km (any origin):
city
Lahore            0.98
Faisalabad        0.98
Multan            0.99
Rawalpindi        0.86
Sialkot           0.96
Rahim Yar Khan    0.84
Islamabad         0.87
Amritsar          0.97
Ferozepur         0.98
Ludhiana          0.97
Jalandhar         0.96
Name: fire_cnt_r150_all, dtype: float64

Filter audit:
                                          step  rows_before  rows_after  removed                                            note
                 Bounding box 26-37 N, 68-79 E      1284773      287323   997450                                                
                 Keep type 0 (vegetation fire)       287323      279968     7355 drops type 2 static sources and type 3 offshore
              Keep confidence n and h (drop l)       279968      238796    41172

**Saved files** (in `Dataset/Exported_Fire/`, not uploaded to GitHub):
- `fire_points_clean.csv`: every type 0 point in the study area, all confidence levels, with the 0/1 dummy columns. Filter on `confidence` to get any version.
- `fire_city_day_long.csv`: every setting, one row per city, day, cone, radius and origin.
- `fire_city_day_wide.csv`: the default setting, one row per city per day (8,030 rows), ready for the merge.
- `fire_filter_audit.csv`: rows before and after each filter (copied into `DECISIONS.md`).

The checks confirm 11 cities, no duplicate city-days, no missing values in the fire columns, and the full 2022-2023 date range.